# Dendrograms: Clustering Without Choosing K

**Computational Text Analysis Pack · Module 06 — Clustering Documents**

*Technique:* Hierarchical clustering; linkage choice and cutting the tree

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/05-unsupervised-structure/06c_dendrograms.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Hierarchical clustering does not ask for k. It merges the two closest documents,
then the next two, and keeps going until everything is one group — and draws the
whole history as a tree.

You then **cut** the tree wherever you like. Cutting is your decision, made
after seeing the structure rather than before.

For DH this is usually the better tool: the tree itself is the result, and it is
the standard figure in stylometry (Module 09).

## The tooling

**scikit-learn** — `TfidfVectorizer` for the document-term matrix, `KMeans` and
`TruncatedSVD` for structure. **scipy**'s `cluster.hierarchy` does the
dendrograms. **gensim** is the alternative once the matrix stops fitting in
memory.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

# Hierarchical clustering is O(n²) in memory. Group documents first — here by
# author — so the tree has tens of leaves rather than thousands.
grouped = (df_colonial.groupby(["work_id", "title", "author"])["text"]
                      .apply(" ".join).reset_index())
grouped = grouped[grouped["text"].str.split().str.len() > 2000]
grouped = grouped.head(30).reset_index(drop=True)
print(f"{len(grouped)} works, each concatenated into one document")

In [ ]:
vec = TfidfVectorizer(token_pattern=r"\w+", min_df=2, max_features=2000,
                      sublinear_tf=True)
X = vec.fit_transform(grouped["text"]).toarray()   # small enough to densify

LINKAGE = "ward"            # ← try "average", "complete", "single"

if LINKAGE == "ward":
    Z = linkage(X, method="ward")
else:
    Z = linkage(pdist(X, metric="cosine"), method=LINKAGE)

labels = [f"{t[:32]} — {a[:18]}" for t, a in zip(grouped["title"], grouped["author"])]
fig, ax = plt.subplots(figsize=(9, 10))
dendrogram(Z, labels=labels, orientation="left", leaf_font_size=8, ax=ax)
ax.set_title(f"Colonial-era works, {LINKAGE} linkage")
ax.set_xlabel("distance at which the branches merge")
plt.tight_layout(); plt.show()

## How to read a dendrogram

- Two leaves joined **low down** are similar
- A branch joining **far right** (or high, if drawn vertically) is a real division
- The **horizontal distance** is what carries meaning; vertical position is only
  drawing order

Look for works by the same author landing on the same branch. When that happens
without the algorithm being told about authorship, the signal is real — and it is
the whole basis of Module 09.

In [ ]:
# Cutting the tree: pick a number of groups after seeing the structure.
N_GROUPS = 4                # ← change this

grouped["group"] = fcluster(Z, N_GROUPS, criterion="maxclust")
for g in sorted(grouped["group"].unique()):
    members = grouped[grouped["group"] == g]
    print(f"group {g} ({len(members)} works):")
    for _, r in members.head(6).iterrows():
        print(f"    {r['title'][:52]:<54}{r['author'][:26]}")
    print()

### 🔧 Your turn 1

Change `LINKAGE` to `"average"`, then `"single"`, and re-run.

**Single linkage** almost always produces a "chain": one long straggling branch
with items added one at a time. It is a known pathology, not a finding. Ward and
average are the defensible choices for text.

## The same tool on the news corpus

Group by category and language instead of by work, and the tree tells you which
kinds of writing resemble each other.

In [ ]:
eng = df[df["language"] == "English"]
groups = (eng.groupby("category")["text"].apply(" ".join).reset_index())
groups = groups[groups["text"].str.split().str.len() > 5000]

vec2 = TfidfVectorizer(token_pattern=r"\w+", min_df=1, max_features=2000,
                       sublinear_tf=True)
X2 = vec2.fit_transform(groups["text"]).toarray()
Z2 = linkage(X2, method="ward")

fig, ax = plt.subplots(figsize=(8, 4.5))
dendrogram(Z2, labels=groups["category"].tolist(), leaf_font_size=10, ax=ax)
ax.set_ylabel("distance")
ax.set_title("News categories clustered by vocabulary")
plt.tight_layout(); plt.show()

### 🔧 Your turn 2

Build the same tree across `language` rather than `category`.

The result will be dominated by which languages are related to each other — a
finding about language families, recovered from newspaper vocabulary. Check it
against what you know: do Hausa (Afro-Asiatic) and Yoruba (Niger-Congo) sit apart?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Single linkage chains, because it merges on the *closest* pair
of points between clusters, so one intermediate document can bridge two unrelated
groups. Complete linkage over-tightens for the mirror-image reason. Ward
minimises within-cluster variance and is the standard choice in stylometry;
average linkage on cosine distance is the usual alternative. State which you
used — the tree changes shape considerably between them.

**Your turn 2.** Related languages cluster together, which is genuinely
impressive for a method that knows nothing about linguistics. But be careful with
the interpretation: TF-IDF on raw word forms is measuring shared orthography and
vocabulary, so it will also group languages that merely share a writing system or
a stock of loanwords. It is a demonstration that the method captures real signal,
not a phylogenetic result.

</details>

## Where this stops

You can group documents with and without choosing k. All of it was
**unsupervised** — no labels used. Next: **Module 07**, where you use the labels
and can therefore measure exactly how right you are.